# Week 5b.1: Reflection

Simple RAG retrieves once and answers from whatever comes back. Nothing in the graph checks whether what came back was any good.

To address this naivety in our implementation, we'll add a step that reads the retrieved passages and judges them against the question. When they fall short, the agent searches again.

That step is called **reflection**.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. Ingestion pipeline

The policy will be split at 400 characters, embedded, stored.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="liquid/lfm-2.5-embedding-350m:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    check_embedding_ctx_length=False,
)

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

embeddings = NVIDIAEmbeddings(model="nvidia/nemotron-3-embed-1b")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

policy = open("husky_policy.md").read()
chunks = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=0).split_text(policy)
store = InMemoryVectorStore.from_texts(chunks, embeddings)

print(f"{len(chunks)} chunks indexed")

## 2. Running example

A customer spilled water on their keyboard and it stopped working.

In [ ]:
from langchain.tools import tool
from support_tools import SYSTEM_PROMPT


@tool
def search_policy(question: str) -> str:
    """Search the Husky Tech written policy and return the passages most relevant
    to a question."""
    hits = store.similarity_search(question, k=2)
    return "\n\n---\n\n".join(h.page_content.strip() for h in hits)


In [ ]:
question = "I spilled water on my keyboard and it stopped working. Can I get a replacement under warranty?"

passages = search_policy.invoke({"question": question})
print(model.invoke(SYSTEM_PROMPT + "\n\nPolicy passages:\n" + passages
                   + f"\n\nCUSTOMER: {question}").text)

It offers to process the claim.

Section 3 of the policy says the warranty *"does not cover accidental damage, liquid damage, or normal wear such as frayed cables."* The document names this exact case.

In [ ]:
print(passages)

Two passages came back.

- One says how an approved claim gets fulfilled, the other says damaged or defective items can be returned during the warranty.
- Both are about warranties. Neither contains the exclusions.

The agent did not answer badly given what it had. It was never given the sentence that mattered.

<img src="figures/rag_pipeline.png" width="820">

## 3. Grading Retrieved Documents

We'll add a step that reads the passages and judges them against the question, before any answer is written.

<img src="figures/reflection_loop.png" width="820">

In [ ]:
GRADE_PROMPT = """A customer asked: {question}

Retrieved policy passages:

{context}

Do these passages contain the policy rules needed to answer this question?
If a rule is referred to but not stated here, it is missing.

Judge the passages, not the customer. The customer may not have mentioned every
detail about their situation, and that is not something retrieval can fix."""

### Custom Grade Data
The judgement will be made by the model and needs to be data, not prose. We'll therefore use `with_structured_output` from W03.

- The `reasoning` field is written before the verdict, so the model has to work out
  which rule settles the question before it commits to an answer. We never read it.
- The `sufficient` field will be `True` if the passages contain every rule needed to answer"
- The `missing` field is what makes the next search better than the last one.
    - If the retrieved content is not sufficient, we'll ask the model to describe what is missing in a few words

In [ ]:
from pydantic import BaseModel, Field


class Grade(BaseModel):
    """Grade whether the retrieved passages answer the customer's question."""

    reasoning: str = Field(
        description="Name the rule that would settle this question, then say "
                    "whether it appears in the passages above")
    sufficient: bool = Field(
        description="True only if the passages contain every rule needed to answer")
    missing: str = Field(
        description="If not sufficient, describe what is missing in a few words")


grader = model.with_structured_output(Grade)


In [ ]:
augmented_prompt = #TODO
print(augmented_prompt)

In [ ]:
grade = grader.invoke(augmented_prompt)
grade

In [ ]:
print(grade.reasoning)

In [ ]:
print(grade.missing)

In [ ]:
grade.sufficient

## 4. Rewriting the query

The customer's words went into the first search. Searching with the same words again returns the same chunks, so the agent has to ask differently.

It will need to write the new query from what the grader said was missing.

**Note that the prompt asks for a question, not keywords.**

 - An embedding model is trained on questions paired with the passages that answer them, so keywords sit badly in that space.

In [ ]:
REWRITE_PROMPT = """A search over a policy document did not return what was needed.

Customer question: {question}
Search just tried: {query}
What is still missing: {missing}

Write a different question to search with, aimed at the part that is missing.
Ask it as a plain question in a full sentence. Do not write a list of keywords.
Reply with the question only."""

In [ ]:
revised_prompt = #TODO
print(revised_prompt)


In [ ]:
result = model.invoke(revised_prompt)
result

In [ ]:
revised_query = result.content
print(revised_query) 

In [ ]:
passages2 = search_policy.invoke({"question": revised_query})
print(passages2)

In [ ]:
augmented_prompt2 = GRADE_PROMPT.format(question=revised_query, 
                                        context=passages2)
grade2 = grader.invoke(augmented_prompt2)
grade2

In [ ]:
grade.sufficient

In [ ]:
FINAL_PROMPT = """{system_prompt}

A check has judged the passages below {grade} to answer this question.

If they were judged insufficient, do not state whether the item is or is not
covered and do not start a claim. Say that you could not find the rule that
settles it, and that a human will need to check.

Policy passages:

{context}

CUSTOMER: {question}"""

In [ ]:
final_prompt = FINAL_PROMPT.format(system_prompt=SYSTEM_PROMPT,
                    grade="sufficient" if grade2.sufficient else "insufficient",
                    context = passages2,
                    question = question)

print(final_prompt)

## 5. State

The loop carries more than messages, so this is a custom state rather than `MessagesState`.

`documents` is annotated with `operator.add`, the same reducer idea as W04b: each retrieval adds to what we have instead of replacing it. Without it the second search throws away the first one's passages.

Everything else replaces, which is the default.

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict


class RetrievalState(TypedDict):
    #TODO

## 6. The nodes

Four short functions. `retrieve` drops passages we already have, so the list does not fill up with the same chunk three times.

In [ ]:
MAX_ATTEMPTS = 3
K = 2


def retrieve(state: RetrievalState) -> dict:
    existing_docs = set(state.get("documents") or [])
    new_docs = [h.page_content for h in store.similarity_search(state["query"], k=K)
             if h.page_content not in existing_docs]
    return {"documents": new_docs, "attempts": state.get("attempts", 0) + 1}


def grade_documents(state: RetrievalState) -> dict:
    augmented_query = GRADE_PROMPT.format(question=state["question"], 
                                context="\n\n".join(state["documents"]))
    verdict = grader.invoke(augmented_query)
    return {"grade": "sufficient" if verdict.sufficient else "insufficient",
            "missing": verdict.missing}


def rewrite_query(state: RetrievalState) -> dict:
    revised_prompt = REWRITE_PROMPT.format(question=state["question"], 
                                            query=state["query"],
                                            missing=state["missing"])
    
    new_query = model.invoke(revised_prompt).text.strip()
    return {"query": new_query}


def generate(state: RetrievalState) -> dict:
    final_prompt = FINAL_PROMPT.format(system_prompt=SYSTEM_PROMPT,
                    grade=state['grade'],
                    context = "\n\n".join(state["documents"]),
                    question = state['question'])
    
    response = model.invoke(final_prompt).text.strip()
    return {"answer": response}

### The routing function 

It returns the name of the next node, annotated with `Literal` so LangGraph knows both destinations, the same way `route` worked in W04a.

The cap is the second condition. A loop that decides when to stop is a loop that can fail to stop.

In [ ]:
def decide(state: RetrievalState) -> Literal["generate", "rewrite_query"]:
    # TODO

## 7. The graph

<img src="figures/reflection_loop.png" width="820">

One edge is new: `rewrite_query -> retrieve` points backwards.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(RetrievalState)
builder.add_node("retrieve", retrieve)
builder.add_node("grade_documents", grade_documents)
builder.add_node("rewrite_query", rewrite_query)
builder.add_node("generate", generate)

#TODO

rag_agent = builder.compile(checkpointer=InMemorySaver())

In [ ]:
from IPython.display import Image, display

display(Image(rag_agent.get_graph().draw_mermaid_png()))

## 8. Test


In [ ]:
config = {"configurable": {"thread_id": "2"}}

result = rag_agent.invoke({"question": question, 
                            "query": question}, config)

print(f"attempts: {result['attempts']}")
print(f"final query: {result['query']}")
print()
print(result["answer"])

### State History

In [ ]:
from pprint import pprint
pprint(rag_agent.get_state(config).values)

In [ ]:
history = list(rag_agent.get_state_history(config))

for i, state in enumerate(history):
    
    # If there's an older state in the history, its ".next" is what just ran
    if i + 1 < len(history):
        chronologically_previous_state = history[i + 1]
        executed_nodes = chronologically_previous_state.next
        # Format the tuple beautifully as a string
        executed_node_name = ", ".join(executed_nodes) if executed_nodes else "__start__"
    else:
        executed_node_name = "__start__" # The very first state in the thread
        
    print(f"Executed Node: {executed_node_name}")
    print(f"Next Node to Run: {state.next}")
    print(f"State Values:\n")
    print(f"\t Question: {state.values.get("question")}\n")
    print(f"\t Query: {state.values.get("query")}\n")
    print(f"\t Docs: {len(state.values.get("documents"))}\n")
    print("\n----\n")

It searches again, finds the exclusions, and says the spill is not covered.

Run it a few times. The attempt count moves and the rewritten query is different each time, because every step here is a model call.

Reflection did not turn an unanswerable question into an answerable one. It let the agent tell the difference between not finding something and there being nothing to find.

## 8. Where this comes from

**Self-Refine** (Madaan et al., 2023): the model produces something, critiques it, revises, repeats.

**Reflexion** (Shinn et al., 2023): the agent writes down what went wrong in words and carries that text into the next attempt. Our `missing` field is a small version of this.

**Self-RAG** (Asai et al., 2023): the retrieval-specific version, where the model is trained to emit tokens saying whether to retrieve and whether what came back is relevant.

NOTE: self-critique works best when the feedback is grounded in something outside the model. Huang et al. (2023) found models frequently fail to spot their own errors without an external signal. Our grader is in between, since it judges retrieved text against a question rather than marking its own answer.

## 9. Things to try

1. Set `MAX_ATTEMPTS = 1` and ask the spill question again. Explain, in terms of the graph, why one attempt is not the same as Tuesday's agent.

2. Delete `operator.add` from the `documents` annotation so it replaces instead of accumulating. Run the spill question several times. What changes about the answer, and why is it worse?

3. Change `REWRITE_PROMPT` to ask for keywords instead of a question. Count the attempts for the spill question before and after.

4. Find another question this policy answers but the loop still gets wrong. Work out whether the grader was wrong about sufficiency, the rewrite was wrong about what to search for, or the document genuinely does not say.

In [ ]:
# your experiments here